In [1]:
#| echo: false
#| output: false

%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from geneinfo.coords import gene_coords
from geneinfo.lift import liftover, get_chain_file

`geneinfo.lift` lifts genomic intervals between assemblies with the UCSC `liftOver` tool, which must be on your `PATH` (install it with `conda install -c bioconda ucsc-liftover`). Chain files are downloaded from UCSC on first use and cached in `~/.cache/geneinfo/chains`.

Intervals use BED conventions: 0-based, half-open coordinates. Chromosome names are translated to the UCSC names used in the chain files (see [Chromosome names](#chromosome-names)).

## Liftover

As an example, take the coordinates of a few human genes:

In [3]:
genes = pd.DataFrame(gene_coords(['BRCA2', 'AMELX', 'TP53', 'XIST', 'FOXP2'], 'hg38'),
                     columns=['chrom', 'start', 'end', 'name'])
genes

,chrom,start,end,name
0,chr13,32315076,32400268,BRCA2
1,chrX,11293412,11309588,AMELX
2,chr17,7668420,7687490,TP53
3,chrX,73820655,73852714,XIST
4,chr7,114086326,114693765,FOXP2


`liftover` returns a copy of the input, with the same index and row order. It adds columns for the coordinates in the other assembly (prefixed with that assembly's name) and a boolean column `mapped`, which is True if the interval lifted. Coordinates are NaN where a lift fails.

In [4]:
liftover(genes, 'hg38', 'papAnu4')

,chrom,start,end,name,papAnu4_chrom,papAnu4_start,papAnu4_end,mapped
0,chr13,32315076,32400268,BRCA2,NaN,NaN,NaN,False
1,chrX,11293412,11309588,AMELX,chrX,8604892.0,8627921.0,True
2,chr17,7668420,7687490,TP53,NaN,NaN,NaN,False
3,chrX,73820655,73852714,XIST,chrX,65214311.0,65247639.0,True
4,chr7,114086326,114693765,FOXP2,NaN,NaN,NaN,False


By default, liftOver requires 95% of an interval's bases to remap (`min_match=0.95`). Whole genes often fail that between species, so lower it for long intervals or divergent species:

In [5]:
liftover(genes, 'hg38', 'papAnu4', min_match=0.5)

,chrom,start,end,name,papAnu4_chrom,papAnu4_start,papAnu4_end,mapped
0,chr13,32315076,32400268,BRCA2,chr17,11326821.0,11413816.0,True
1,chrX,11293412,11309588,AMELX,chrX,8604892.0,8627921.0,True
2,chr17,7668420,7687490,TP53,chr16,7371046.0,7389350.0,True
3,chrX,73820655,73852714,XIST,chrX,65214311.0,65247639.0,True
4,chr7,114086326,114693765,FOXP2,chr3,136782265.0,137385029.0,True


## Reciprocal liftover

With `reciprocal=True`, the lifted intervals are also lifted back to the original assembly. An interval maps reciprocally if it lifts back to exactly its original coordinates. Intervals that pass this test correspond one-to-one between the two assemblies.

This adds columns for the coordinates lifted back (prefixed with the original assembly's name and `_back`) and a boolean column `reciprocal`. If an interval is `mapped` but not `reciprocal`, it either did not lift back (its `hg38_back_*` coordinates are NaN) or it lifted back to different coordinates.

In [6]:
lifted = liftover(genes, 'hg38', 'papAnu4', reciprocal=True)
lifted

,chrom,start,end,name,papAnu4_chrom,papAnu4_start,papAnu4_end,hg38_back_chrom,hg38_back_start,hg38_back_end,mapped,reciprocal
0,chr13,32315076,32400268,BRCA2,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1,chrX,11293412,11309588,AMELX,chrX,8604892.0,8627921.0,NaN,NaN,NaN,True,False
2,chr17,7668420,7687490,TP53,NaN,NaN,NaN,NaN,NaN,NaN,False,False
3,chrX,73820655,73852714,XIST,chrX,65214311.0,65247639.0,chrX,73820655.0,73852714.0,True,True
4,chr7,114086326,114693765,FOXP2,NaN,NaN,NaN,NaN,NaN,NaN,False,False


With a lower `min_match`, all five genes map reciprocally:

In [7]:
lifted = liftover(genes, 'hg38', 'papAnu4', reciprocal=True, min_match=0.5)
lifted[['name', 'papAnu4_chrom', 'papAnu4_start', 'papAnu4_end', 'mapped', 'reciprocal']]

,name,papAnu4_chrom,papAnu4_start,papAnu4_end,mapped,reciprocal
0,BRCA2,chr17,11326821.0,11413816.0,True,True
1,AMELX,chrX,8604892.0,8627921.0,True,True
2,TP53,chr16,7371046.0,7389350.0,True,True
3,XIST,chrX,65214311.0,65247639.0,True,True
4,FOXP2,chr3,136782265.0,137385029.0,True,True


Keep only the reciprocally mapped intervals:

In [8]:
lifted.loc[lifted.reciprocal]

,chrom,start,end,name,papAnu4_chrom,papAnu4_start,papAnu4_end,hg38_back_chrom,hg38_back_start,hg38_back_end,mapped,reciprocal
0,chr13,32315076,32400268,BRCA2,chr17,11326821.0,11413816.0,chr13,32315076.0,32400268.0,True,True
1,chrX,11293412,11309588,AMELX,chrX,8604892.0,8627921.0,chrX,11293412.0,11309588.0,True,True
2,chr17,7668420,7687490,TP53,chr16,7371046.0,7389350.0,chr17,7668420.0,7687490.0,True,True
3,chrX,73820655,73852714,XIST,chrX,65214311.0,65247639.0,chrX,73820655.0,73852714.0,True,True
4,chr7,114086326,114693765,FOXP2,chr3,136782265.0,137385029.0,chr7,114086326.0,114693765.0,True,True


## Positions

To lift single positions, such as SNPs, name the column holding them with `pos_col`. The interval columns are then ignored. Positions are 0-based by default, as BED starts; use `pos_base=1` for 1-based positions, as in VCF files. Lifted positions are reported in the same base.

In [9]:
snps = pd.DataFrame({'chrom': ['chr13', 'chrX', 'chr7'],
                     'pos': [32315500, 73821000, 114100000]})
liftover(snps, 'hg38', 'papAnu4', pos_col='pos', pos_base=1, reciprocal=True)

,chrom,pos,papAnu4_chrom,papAnu4_pos,hg38_back_chrom,hg38_back_pos,mapped,reciprocal
0,chr13,32315500,chr17,11327244.0,chr13,32315500.0,True,True
1,chrX,73821000,chrX,65214655.0,chrX,73821000.0,True,True
2,chr7,114100000,chr3,136796001.0,chr7,114100000.0,True,True


## Lifting through an intermediate assembly

UCSC has no chain file for many pairs of assemblies, for example from the T2T assembly (`hs1`) to baboon (`papAnu4`). If there are chains to and from an intermediate assembly, name it with `assembly_over`. Intervals are then lifted in two steps, from `hs1` to `hg38` to `papAnu4`, and back the same way with `reciprocal=True`. The output is the same as for a direct lift:

In [10]:
t2t = pd.DataFrame({'chrom': ['chrX', 'chrX'], 'start': [127743009, 41198220],
                    'end': [127744009, 41199220]})
liftover(t2t, 'hs1', 'papAnu4', assembly_over='hg38', reciprocal=True)

,chrom,start,end,papAnu4_chrom,papAnu4_start,papAnu4_end,hs1_back_chrom,hs1_back_start,hs1_back_end,mapped,reciprocal
0,chrX,127743009,127744009,chrX,118238693.0,118239695.0,chrX,127743009.0,127744009.0,True,True
1,chrX,41198220,41199220,chrX,39140748.0,39141749.0,chrX,41198220.0,41199220.0,True,True


Each step applies `min_match`, so an interval can be lost in either step. A list of assemblies lifts through each of them in turn.

## Chromosome names

Chain files use UCSC chromosome names such as `chrX`. `liftover` translates other names, such as RefSeq (`NC_000023.11`), GenBank (`CM000685.2`) or Ensembl-style (`X`), using the UCSC chromosome alias table of the source assembly. The lifted coordinates always use UCSC names, and the input columns are left unchanged:

In [11]:
df = pd.DataFrame({'chrom': ['NC_000023.11', 'X', 'chrX'],
                   'start': 73820655, 'end': 73821655})
liftover(df, 'hg38', 'papAnu4')

,chrom,start,end,papAnu4_chrom,papAnu4_start,papAnu4_end,mapped
0,NC_000023.11,73820655,73821655,chrX,65214311.0,65215315.0,True
1,X,73820655,73821655,chrX,65214311.0,65215315.0,True
2,chrX,73820655,73821655,chrX,65214311.0,65215315.0,True


Names that are not in the alias table are used as given, and a warning lists them. A common cause is giving the wrong source assembly: `NC_060947.1` is chrX in the T2T assembly (`hs1`), not in hg38. Pass `chrom_aliases=False` to turn off the translation.

## Performance

liftOver handles roughly 2,000 intervals per second per process. `liftover` lifts each distinct interval (or position) only once, however often it occurs, and splits large inputs across parallel liftOver processes. By default, it uses all available cores (`SLURM_CPUS_PER_TASK` in a SLURM job), but no more processes than fit in memory, because each process loads the whole chain file (up to about 1 GB). Set `n_jobs` to choose the number of processes yourself.

## Input and chain files

`liftover` also takes the path to a BED file, read with standard BED column names (`chrom`, `start`, `end`, `name`, ...). If a data frame uses other names for its coordinate columns, pass them with `chrom_col`, `start_col` and `end_col`.

`get_chain_file` returns the path to a cached UCSC chain file, downloading it if needed:

In [12]:
path = get_chain_file('hg38', 'papAnu4')
path.name

'hg38ToPapAnu4.over.chain.gz'

To use your own chain files, pass them explicitly. The assembly names are then only used to name the output columns:

```python
liftover(genes, 'hg38', 'papAnu4', reciprocal=True,
         chain_file='hg38ToPapAnu4.over.chain.gz',
         back_chain_file='papAnu4ToHg38.over.chain.gz')
```